In [ ]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd().resolve()

def find_release_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'Step0_Data').is_dir() and (candidate / 'Step1_FeaturePool').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate standalone release root from the current working directory')

release_root = find_release_root(cwd)
step_root = release_root / 'Step2_WDI_Features'
input_path = release_root / 'Step1_FeaturePool' / '2_Results' / '1_selected_features_data.csv'
diagnostic_path = step_root / '2_Results' / '0_diagnostic_report.csv'
output_path = step_root / '2_Results' / '1_imputed_features_data.csv'
targets = ['IW_t', 'AW_t', 'CDW_t', 'MSW_t']
meta_cols = ['Country Code', 'Year', 'Country Name', 'IncomeGroup', 'Region']

In [ ]:
df = pd.read_csv(input_path)
features = [col for col in df.columns if col not in targets + meta_cols]
feature_report = pd.DataFrame({
    'feature': features,
    'missing_rows': [int(df[col].isna().sum()) for col in features],
    'missing_pct': [float(df[col].isna().mean() * 100.0) for col in features],
    'island_countries': [int((df.groupby('Country Code')[col].apply(lambda s: int(s.isna().sum())) == 35).sum()) for col in features],
})
diagnostic_path.parent.mkdir(parents=True, exist_ok=True)
feature_report.to_csv(diagnostic_path, index=False)
feature_report.sort_values('missing_rows', ascending=False).head(10)

In [ ]:
df = df.sort_values(['Country Code', 'Year']).reset_index(drop=True)
original_targets = df[targets].copy()
for col in features:
    df[col] = df.groupby('Country Code')[col].transform(
        lambda series: series.interpolate(method='linear').ffill().bfill() if series.isna().sum() < len(series) else series
    )
df[targets] = original_targets
df[targets].isna().sum().to_dict()

In [ ]:
df.to_csv(output_path, index=False)
output_path